# VIO quality -- offline VINS pose vs FC EKF/GPS truth

Per-flight comparison of the **deterministically regenerated** VINS pose against the
flight controller's EKF/GPS state, using the vetted `analysis/vio_ekf_compare.py`.

- **Input:** the FC `.bin` (parameter `input_file`). The sibling VINS pose
  `*.vinspose.csv` + its `*.vinspose.polisher.json` provenance sidecar (from
  `vio-offline-runner`) are found in the same dataset dir.
- **Built on flight-analysis:** FC-side facts (RTK status, vibration, EKF health) are
  *consumed* from that flight's `manifest.json` -- not re-derived here. Run
  flight-analysis first.
- **Reproducible:** the pose is byte-deterministic for a given estimator source SHA +
  fixture + config (recorded in the sidecar). Every number in the AGENT-READABLE block
  comes from the same `compare()` call that draws the figure, so the JSON and the plot
  cannot disagree.

This notebook does **not** try to establish *why* VINS diverges -- only to measure,
reproducibly, how the deployed estimator's pose relates to GPS/EKF truth. Causation
stays in `vio-quality-experiments.md`.

In [ ]:
# parameters -- papermill overrides `input_file` per flight
input_file = "/home/jovyan/datasets/flights/rekon10/260705-handheld-noarm/1980-01-06 10-14-19.bin"
debug = False

In [ ]:
import os, sys, json, hashlib, datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# vio_ekf_compare lives in coordinator/analysis alongside this notebook. Add whichever
# of cwd / cwd/analysis / parent actually contains it (covers running from the analysis
# dir, the repo root, or a papermill workdir one level down).
_here = Path.cwd()
for _cand in (_here, _here / "analysis", _here.parent):
    if (_cand / "vio_ekf_compare.py").exists():
        sys.path.insert(0, str(_cand)); break
import vio_ekf_compare as vc

LOG = Path(input_file)
D = LOG.parent
print("flight dir:", D)
print("FC log    :", LOG.name)

In [ ]:
# Resolve the sibling artifacts: exactly one VINS pose per flight dir, its provenance
# sidecar, and the flight-analysis manifest we build on.
pose_csvs = sorted(D.glob("*.vinspose.csv"))
assert len(pose_csvs) == 1, f"expected exactly one *.vinspose.csv in {D}, found {len(pose_csvs)}"
POSE = pose_csvs[0]
SIDECAR = POSE.with_suffix(".polisher.json")   # <stem>.vinspose.polisher.json
FA_MANIFEST = D / "manifest.json"
print("VINS pose :", POSE.name)
print("sidecar   :", SIDECAR.name, "(exists)" if SIDECAR.exists() else "(MISSING)")
print("flight-analysis manifest:", "present" if FA_MANIFEST.exists() else "MISSING")

In [ ]:
# --- ASSUMPTIONS: fail loudly before trusting any comparison ---
issues = []

# 1. Provenance must be present and match the actual CSV (refuse untracked/altered pose).
assert SIDECAR.exists(), f"no provenance sidecar {SIDECAR.name} -- pose is untracked, refusing"
prov = json.loads(SIDECAR.read_text())
est_sha = prov["instrument"]["sha"]
csv_sha = hashlib.sha256(POSE.read_bytes()).hexdigest()
assert prov["result"][0]["sha256"] == csv_sha, \
    "vinspose.csv sha256 != sidecar -- the pose file changed since it was regenerated"
if est_sha == "unknown":
    issues.append("VINS pose provenance sha is 'unknown' (local/dirty build, not a pushed image)")

# 2. We build ON flight-analysis -- its manifest must exist and be for THIS log.
assert FA_MANIFEST.exists(), f"flight-analysis manifest.json missing in {D} -- run flight-analysis first"
fa = json.loads(FA_MANIFEST.read_text())
assert Path(fa["input_file"]).name == LOG.name, \
    f"manifest.json is for {Path(fa['input_file']).name}, not {LOG.name}"

# 3. FC truth quality (consumed from flight-analysis, not re-derived).
gps_max = fa.get("gps_status_max", 0)
if gps_max < 3:
    issues.append(f"GPS never reached 3D fix (flight-analysis gps_status_max={gps_max}) -- truth unreliable")
if fa.get("ekf_errors", 0) > 0:
    issues.append(f"flight-analysis reports {fa['ekf_errors']} EKF variance error(s) -- truth may be degraded")
for w in fa.get("assumption_warnings", []):
    issues.append(f"[flight-analysis] {w}")

print("=== ASSUMPTIONS ===")
for i in issues:
    print("  WARNING:", i)
if not issues:
    print("  all ok")
print(f"  estimator source sha: {est_sha}")
print(f"  GPS status max: {gps_max}  (3=3D, 4=DGPS, 5=RTK float, 6=RTK fixed)")

## FC-side truth (consumed from flight-analysis)

These are the vetted facts from the canonical flight-analysis notebook -- the same
numbers that appear in its PDF. This VIO analysis leans on them; **eyeball them against
`flight-analysis-<stem>.pdf` in this dir** (the machine-readable JSON and the human
figures should agree -- that consistency is what lets us trust the number).

In [ ]:
print("FC facts (from flight-analysis manifest.json):")
for k in ("duration_s", "armed_s", "gps_status_max", "vibe_max_ms2", "clip_total",
          "ekf_errors", "status"):
    print(f"  {k:16s}: {fa.get(k)}")
if fa.get("assumption_warnings"):
    print("  flight-analysis warnings:")
    for w in fa["assumption_warnings"]:
        print("    -", w)

## VINS vs FC EKF/GPS comparison

`vio_ekf_compare.compare()` time-aligns VINS to the FC via angular-rate cross-correlation
(on the pre-divergence segment), fits a similarity transform (Umeyama; scale != 1 exposes
VINS mis-scaling), and reports ATE over the usable window plus how long VINS stayed within
1 m of EKF before running away. The figure below and the metrics come from this one call.

In [ ]:
run_name = D.name
metrics = vc.compare(str(POSE), str(LOG), run_name=run_name, make_plot=True)
plt.show()
print(json.dumps({k: (float(v) if isinstance(v, np.floating) else v)
                  for k, v in metrics.items()}, indent=2, default=str))

In [ ]:
# AGENT-READABLE STATS -- consolidated JSON, from the SAME compare() call that drew the
# figure above. Written next to the flight (NB_OUTPUT_DIR override for the cluster job).
out_dir = Path(os.environ.get("NB_OUTPUT_DIR", str(D)))

# self-consistency (fail loudly if the metrics are internally impossible)
m = metrics
assert m["ate_max_m"] is None or m["ate_max_m"] >= m["ate_rmse_m"] >= 0, "ATE ordering impossible"
assert -1.0 <= m["align_ncc"] <= 1.0, "alignment NCC out of range"
assert m["umeyama_scale"] > 0, "non-positive scale"

agent = {
    "notebook": "vio-quality.ipynb",
    "input_file": str(LOG),
    "vinspose_csv": POSE.name,
    "run_at": datetime.datetime.utcnow().isoformat() + "Z",
    "estimator_provenance": {              # who produced the pose (from the sidecar)
        "sha": prov["instrument"]["sha"],
        "image": prov["instrument"]["image"],
        "vins_fusion_sha": prov["instrument"]["vins_fusion_sha"],
        "vinspose_sha256": prov["result"][0]["sha256"],
        "pose_rows": prov["result"][0].get("pose_rows"),
        "config_sha256": next((o["sha256"] for o in prov["object"] if o["name"].endswith(".yaml")), None),
    },
    "flight_analysis": {                   # consumed FC truth (not re-derived)
        "gps_status_max": fa.get("gps_status_max"),
        "vibe_max_ms2": fa.get("vibe_max_ms2"),
        "clip_total": fa.get("clip_total"),
        "ekf_errors": fa.get("ekf_errors"),
        "status": fa.get("status"),
    },
    "comparison": {k: (float(v) if isinstance(v, np.floating) else v) for k, v in m.items()},
    "assumption_warnings": issues,
    "status": "ok" if not issues else "warnings",
}
(out_dir / "vio-quality.json").write_text(json.dumps(agent, indent=2, default=str) + "\n")
print("wrote", out_dir / "vio-quality.json")
print(json.dumps(agent, indent=2, default=str))

## Reading this (measured vs not proven)

- **Measured (reproducible):** the alignment NCC, Umeyama scale, ATE over the usable
  window, and how many seconds VINS tracked within 1 m before diverging. All keyed to a
  specific estimator source SHA + fixture + config (see `estimator_provenance`).
- **A high NCC** means the VINS trajectory *shape* follows the EKF early on -- the
  estimator initialises and tracks, it is not garbage-out.
- **Scale != 1** is VINS mis-scaling relative to metric truth; **usable_track_s** is when
  it stops being usable.
- **Not established here:** *why* it diverges (IMU/extrinsic/calibration/vibration/cycles).
  That lives in `vio-quality-experiments.md`. The handheld (motors-off) run is the clean
  control; compare its numbers to the armed run there, not by eye here.